# Funnel Analysis & Drop-Off Detection

## Objective
Analyze the HirePulse recruitment funnel, calculate stage-wise drop-off and completion rates, identify the biggest bottleneck, estimate illustrative business impact, and define actionable recommendations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

INPUT_FILE = Path("../data/recruitment_stages.csv")
OUTPUT_DIR = Path("../output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
df.head()

## 1. Define Funnel Stages

The recruitment funnel is defined using the sequential stages available in the dataset:

1. Applied
2. Screening
3. Technical Interview
4. HR Interview
5. Offer Sent
6. Joined

In [ ]:
stage_order = ["Applied", "Screening", "Technical Interview", "HR Interview", "Offer Sent", "Joined"]

stages = {}

for stage in stage_order:
    stages[stage] = df.loc[df["stage_name"] == stage, "candidate_id"].nunique()

stage_counts = pd.DataFrame({"stage": stage_order, "candidates": [stages[s] for s in stage_order]})
stage_counts

## 2. Drop-Off Analysis

For each transition:

- **Users Lost** = Users Before − Users After
- **Drop Rate** = Users Lost / Users Before × 100
- **Completion Rate** = Users After / Users Before × 100

In [ ]:
drop_off = []

for i in range(len(stage_order) - 1):
    from_stage = stage_order[i]
    to_stage = stage_order[i + 1]
    users_before = stages[from_stage]
    users_after = stages[to_stage]
    users_lost = users_before - users_after
    completion_rate = (users_after / users_before) * 100 if users_before else 0
    drop_rate = (users_lost / users_before) * 100 if users_before else 0

    drop_off.append({
        "from_stage": from_stage,
        "to_stage": to_stage,
        "users_before": users_before,
        "users_after": users_after,
        "users_lost": users_lost,
        "completion_rate": completion_rate,
        "drop_rate": drop_rate
    })

funnel_df = pd.DataFrame(drop_off)
funnel_df

In [ ]:
biggest_drop = funnel_df.loc[funnel_df["users_lost"].idxmax()]
highest_drop_rate = funnel_df.loc[funnel_df["drop_rate"].idxmax()]

starting_users = stages[stage_order[0]]
final_users = stages[stage_order[-1]]
overall_conversion = (final_users / starting_users) * 100

action_summary = {
    "Biggest absolute drop": f"{biggest_drop["from_stage"]} -> {biggest_drop["to_stage"]}",
    "Users lost": int(biggest_drop["users_lost"]),
    "Drop rate": f"{biggest_drop["drop_rate"]:.1f}%",
    "Highest drop-rate transition": f"{highest_drop_rate["from_stage"]} -> {highest_drop_rate["to_stage"]}",
    "Overall conversion": f"{overall_conversion:.1f}%"
}

action_summary

## 3. Funnel Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.bar(stage_order, [stages[s] for s in stage_order])

ax.set_title("Recruitment Funnel: Candidate Volume by Stage", fontsize=14, fontweight="bold")
ax.set_xlabel("Recruitment Stage")
ax.set_ylabel("Candidates")
ax.set_ylim(0, max(stages.values()) * 1.15)

for bar, stage in zip(bars, stage_order):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(stages[stage]),
        ha="center",
        va="bottom",
        fontweight="bold"
    )

plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "funnel_chart.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Business Impact

The dataset does not contain an actual revenue-per-hire or customer LTV value. Therefore, **00 per successful conversion** is used only as an illustrative scenario value, consistent with the assignment example.

In [ ]:
revenue_per_customer = 100

impact_df = funnel_df.copy()
impact_df["revenue_impact"] = impact_df["users_lost"] * revenue_per_customer
impact_df["priority"] = impact_df["revenue_impact"].apply(
    lambda x: "HIGH" if x > 1000 else ("MEDIUM" if x > 500 else "LOW")
)

impact_df[["from_stage", "to_stage", "users_lost", "drop_rate", "revenue_impact", "priority"]].sort_values(
    "revenue_impact", ascending=False
)

## 5. Recommendation

The primary bottleneck is **Offer Sent → Joined**, where 20 candidates are lost and the drop rate is 28.6%.

### Root-cause hypotheses
- Unclear offer instructions or expectations
- Process or documentation friction
- Scheduling or communication delays
- Insufficient information before joining
- Timing-related delays

### Recommended actions
1. Investigate candidate feedback around this transition.
2. Review the process for unnecessary complexity or delays.
3. Test a simplified version of the step.
4. Monitor completion and drop rates after the change.
5. Compare the revised funnel against the baseline.

### Success criteria
- Increase completion rate at the bottleneck stage.
- Reduce the stage-specific drop rate.
- Confirm that improvement persists across multiple cohorts.
- Monitor final Joined conversion for downstream impact.

## 6. Comparing Funnels Across Time Periods or Segments

The same funnel methodology can be applied separately by month, quarter, department, candidate source, or another available segment. Comparing stage counts, completion rates, and drop rates across groups can reveal whether a bottleneck is concentrated in a particular cohort or segment.

## Conclusion

The recruitment funnel starts with 100 candidates and ends with 50 Joined candidates, producing a 50.0% overall conversion rate. The largest observed bottleneck is **Offer Sent → Joined**, with 20 candidates lost and a 28.6% drop rate. This stage should be investigated using candidate feedback and process-level analysis, with completion rate and final Joined conversion used as measurable success criteria.